# 🪷 CHÙA ONLINE - TIKTOK LIVE INTERACTIVE GAME
### HỆ THỐNG PHÁT SÓNG TƯƠNG TÁC TỰ ĐỘNG 24/7 TRÊN GOOGLE COLAB

Notebook này hỗ trợ cả 2 hình thức phát sóng:
- **Cách 1:** Colab làm máy chủ -> cấp link web 3D để bạn xem hoặc dán vào OBS trên máy tính.
- **Cách 2 (Treo máy 24/7 tự động):** Colab tự tạo **Màn hình ảo (Virtual Display 1080x1920)** và tự livestream thẳng lên TikTok bằng **Stream Key** mà không cần mở OBS trên máy tính!


In [ ]:
# @title 📥 Bước 1: Tải mã nguồn & Cài đặt môi trường
import os
import subprocess

print("⏳ Đang tải mã nguồn và cài đặt dependencies...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)

%cd /content/app/connector
!npm install --silent

# Cài đặt cloudflared (để mở web) và xvfb (màn hình ảo 1080x1920 cho live 24/7)
!apt-get update -qq && apt-get install -y -qq xvfb ffmpeg > /dev/null 2>&1
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✅ CÀI ĐẶT HOÀN TẤT!")


In [ ]:
# @title 🚀 Bước 2: Khởi chạy Server Chùa Online & Lấy Link Xem 3D
# @markdown Chọn chế độ chạy và bấm nút Play để khởi động hệ thống:

Che_Do_Chay = "Mock Mode (Test giả lập sự kiện)" # @param ["Mock Mode (Test giả lập sự kiện)", "TikTok Live Thật"]
TikTok_Username = "zuno_live" # @param {type:"string"}

import time
import os
import re
import subprocess
from IPython.display import display, HTML

# 1. Dọn dẹp tiến trình cũ
!pkill -f node > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# 2. Cấu hình biến môi trường
is_mock = "true" if "Mock" in Che_Do_Chay else "false"
env_vars = os.environ.copy()
env_vars["MOCK_MODE"] = is_mock
env_vars["TIKTOK_USERNAME"] = TikTok_Username
env_vars["HTTP_PORT"] = "3000"
env_vars["WS_PORT"] = "8080"

print(f"⏳ Đang khởi động Server Node.js (Chế độ: {'GIẢ LẬP MOCK' if is_mock == 'true' else f'KÊNH @{TikTok_Username}'})...")
with open("/tmp/server.log", "w") as f_log:
    server_process = subprocess.Popen(["node", "src/index.js"], cwd="/content/app/connector", env=env_vars, stdout=f_log, stderr=subprocess.STDOUT)

time.sleep(3)

# Kiểm tra xem Server Node có bị thoát không
if server_process.poll() is not None:
    print("❌ Lỗi: Server Node.js bị dừng. Dưới đây là nội dung log lỗi:")
    !cat /tmp/server.log
else:
    print("✅ Server Node.js đang chạy ổn định!")
    
    # 3. Lấy đường link truy cập công khai
    public_url = None
    
    # Thử lấy link qua Cloudflare Tunnel (non-blocking)
    print("🌐 Đang tạo đường dẫn HTTPS công khai...")
    with open("/tmp/cloudflared.log", "w") as cf_log:
        tunnel_process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=cf_log, stderr=subprocess.STDOUT)
    
    for _ in range(12):
        time.sleep(1)
        if os.path.exists("/tmp/cloudflared.log"):
            with open("/tmp/cloudflared.log", "r") as cf_read:
                content = cf_read.read()
                match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
                if match:
                    public_url = match.group(0)
                    break
    
    # Nếu Cloudflare chưa kịp cấp link, dùng cổng proxy nội bộ của Colab
    if not public_url:
        try:
            from google.colab.output import eval_js
            public_url = eval_js("google.colab.kernel.proxyPort(3000)").rstrip('/')
            print("ℹ️ Đang sử dụng đường dẫn Colab Proxy nội bộ.")
        except Exception:
            pass

    if public_url:
        client_url = f"{public_url}/client"
        dashboard_url = public_url
        html_banner = f"""
        <div style="background: #1a0f0a; border: 2px solid #f1c40f; border-radius: 12px; padding: 20px; color: #fff; font-family: sans-serif;">
            <h2 style="color: #f1c40f; margin-top: 0;">🎉 SERVER CHÙA ONLINE ĐÃ SẴN SÀNG!</h2>
            <p>Bấm vào các nút bên dưới để mở giao diện:</p>
            <div style="margin: 15px 0;">
                <a href="{client_url}" target="_blank" style="display: inline-block; background: #e67e22; color: #fff; padding: 12px 20px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 15px; margin-right: 10px;">🪷 MỞ MÀN HÌNH CHÙA 3D (Tỉ lệ 9:16)</a>
                <a href="{dashboard_url}" target="_blank" style="display: inline-block; background: #27ae60; color: #fff; padding: 12px 20px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 15px;">🎮 BẢNG ĐIỀU KHIỂN & TEST NÚT BẤM</a>
            </div>
            <p style="color: #ccc; font-size: 13px;">
                💡 <em>Để phát live bằng Stream Key trực tiếp từ Colab: Hãy chuyển xuống <strong>Bước 3</strong> bên dưới!</em>
            </p>
        </div>
        """
        display(HTML(html_banner))
    else:
        print("⚠️ Không lấy được link công khai. Server vẫn đang chạy tại cổng local 3000.")
        !cat /tmp/server.log


---
## 🔴 PHẦN 2: TỰ ĐỘNG PHÁT LIVESTREAM LÊN TIKTOK TỪ COLAB (KHÔNG CẦN OBS)
Phần này sẽ khởi tạo **màn hình ảo 1080x1920** ngay bên trong Google Colab, mở Chùa 3D và dùng **FFmpeg** đẩy thẳng luồng video + âm thanh chuông mõ lên kênh TikTok của bạn!


In [ ]:
# @title 🎥 Bước 3: Nhập Stream Key & Khởi chạy Livestream Tự Động 24/7 { display-mode: "form" }
# @markdown Nhập thông tin lấy từ TikTok Live Studio hoặc mục Phát Live trên TikTok:

TikTok_Server_URL = "rtmp://live-push.tiktok.com/live/" # @param {type:"string"}
TikTok_Stream_Key = "" # @param {type:"string"}
FPS = 30 # @param [25, 30, 60]
Video_Bitrate = "3500k" # @param ["2500k", "3500k", "4500k", "6000k"]

import subprocess
import time

if not TikTok_Stream_Key.strip():
    print("⚠️ Vui lòng điền TikTok_Stream_Key của bạn trước khi bấm chạy!")
else:
    # Tắt màn hình ảo và ffmpeg cũ nếu có
    !pkill -f Xvfb > /dev/null 2>&1
    !pkill -f chrome > /dev/null 2>&1
    !pkill -f ffmpeg > /dev/null 2>&1
    time.sleep(1)

    print("🖥️ 1. Đang khởi tạo Màn hình ảo (Virtual Display) 1080x1920...")
    xvfb_proc = subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1080x1920x24"])
    time.sleep(1.5)

    print("🌐 2. Đang mở Màn hình Chùa 3D trên màn hình ảo...")
    chrome_env = os.environ.copy()
    chrome_env["DISPLAY"] = ":99"
    chrome_cmd = [
        "google-chrome",
        "--no-sandbox",
        "--disable-gpu",
        "--disable-dev-shm-usage",
        "--window-size=1080,1920",
        "--window-position=0,0",
        "--kiosk",
        "--autoplay-policy=no-user-gesture-required",
        "http://localhost:3000/client"
    ]
    chrome_proc = subprocess.Popen(chrome_cmd, env=chrome_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)

    # Ghép địa chỉ RTMP đầy đủ
    rtmp_target = f"{TikTok_Server_URL.rstrip('/')}/{TikTok_Stream_Key.lstrip('/')}"
    audio_file = "/content/app/assets/meditation_chant.mp3"

    print(f"📡 3. Đang phát luồng Livestream trực tiếp lên TikTok Live...")
    ffmpeg_cmd = [
        "ffmpeg", "-y",
        "-f", "x11grab",
        "-draw_mouse", "0",
        "-framerate", str(FPS),
        "-video_size", "1080x1920",
        "-i", ":99.0",
        "-stream_loop", "-1",
        "-i", audio_file,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-tune", "zerolatency",
        "-b:v", Video_Bitrate,
        "-maxrate", Video_Bitrate,
        "-bufsize", "6000k",
        "-pix_fmt", "yuv420p",
        "-g", str(FPS * 2),
        "-c:a", "aac",
        "-b:a", "128k",
        "-ar", "44100",
        "-f", "flv",
        rtmp_target
    ]
    ffmpeg_log = open("/tmp/ffmpeg_stream.log", "w")
    stream_proc = subprocess.Popen(ffmpeg_cmd, stdout=ffmpeg_log, stderr=subprocess.STDOUT)
    time.sleep(3)

    if stream_proc.poll() is None:
        print("🔴🔴🔴 LIVESTREAM ĐANG PHÁT TRỰC TIẾP TRÊN KÊNH TIKTOK CỦA BẠN!")
        print("👉 Bạn có thể vào kênh TikTok kiểm tra luồng live ngay bây giờ.")
    else:
        print("❌ Lỗi khi phát luồng. Kiểm tra log:")
        !cat /tmp/ffmpeg_stream.log


In [ ]:
# @title 📊 Bước 4: Kiểm tra Tốc độ & Trạng thái Luồng Phát TikTok
!tail -n 15 /tmp/ffmpeg_stream.log 2>/dev/null || echo "Chưa có log livestream"


In [ ]:
# @title ⏹️ Bước 5: Dừng toàn bộ Livestream & Tắt máy
!pkill -f ffmpeg
!pkill -f chrome
!pkill -f Xvfb
!pkill -f node
!pkill -f cloudflared
print("🛑 ĐÃ DỪNG TOÀN BỘ PHIÊN LIVESTREAM AN TOÀN!")
